# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # This is an object, not a dictionary

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets, their @id, and their fields

record_sets = list(metadata.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for recset in record_sets:
    print(f"- RecordSet name: {recset.name}, @id: {recset.id}")
    if recset.fields:
        print("  Fields:")
        for field in recset.fields:
            # Each field has its own id and name
            print(f"      - Field name: {field.name}, @id: {field.id}")
            # Fields may have column links
            if getattr(field, 'columns', None):
                print("        Columns:")
                for col in field.columns:
                    print(f"          - Column name: {col.name}, @id: {col.id}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# List of all RecordSet @ids
record_set_ids = [r.id for r in metadata.record_sets]
print(f"RecordSets for extraction: {record_set_ids}")
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))  # Each is a dict by field @id
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"\nColumns in RecordSet '{record_set_id}':")
    print(dataframes[record_set_id].columns.tolist())

# Choose the first record set for demonstration
if record_set_ids:
    first_record_set = record_set_ids[0]
    print(f"\nSample data from RecordSet '{first_record_set}':")
    display(dataframes[first_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Example: Filter and normalize a numeric field in the protein data
# We'll choose common protein data fields, but you should adjust these based on available field @ids from Section 2

# Use the first record set
df = dataframes[first_record_set]

# Guess likely field IDs for EDA (check actual @ids above); common are MW or peptides_count
# For demonstration, list column candidates
print("Columns available for EDA:")
print(df.columns.tolist())

# Example: Try to select a numeric field, e.g., with 'MW' or 'peptide' in name
import re
numeric_field_candidates = [col for col in df.columns if re.search(r'(mw|peptide|abundance|coverage|count|total|number|mass|score|intensity)', col, re.I)]
print(f"Numeric field candidates: {numeric_field_candidates}")

if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    print(f"Using field '{numeric_field}' for numeric analysis.")
    # Remove non-numeric or missing entries
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the chosen numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean())/filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping, e.g., by a categorical field (suggestion: 'description', 'accession', or similar)
    group_field_candidates = [col for col in df.columns if re.search(r'(description|accession|group|category|class|type|id|uniprot)', col, re.I)]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        print(f"Grouping by '{group_field}':")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        display(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("No numeric field candidates found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field_candidates:
    # Histogram of the selected numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    # Boxplot by group (if available)
    if group_field_candidates:
        plt.figure(figsize=(12, 5))
        top_groups = df[group_field].value_counts().head(10).index
        sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_groups)])
        plt.title(f'{numeric_field} distribution by {group_field} (Top 10)')
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 mass spectrometry dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library:

- Loaded dataset metadata and records directly from the Croissant schema URL.
- Identified available record sets and field `@id`s using the schema's metadata for robust, schema-aligned access.
- Loaded tabular data for analysis and performed exploratory data analysis (EDA) tasks, such as filtering records by numeric criteria, normalization, and groupwise summarization, using field `@id` as reference throughout.
- Created initial visualizations (histogram and boxplot) for selected numeric fields, grouped by categorical fields when available.

This reproducible approach can be adapted to any Croissant-annotated dataset, accelerating FAIR data exploration and robust feature extraction for machine learning workflows.